<!-- colab-badge -->
[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Zeshengliu213/ai-for-materials/blob/main/01%20-%20Basics/01d_rdkit_basics.ipynb)

# 01d · 用 RDKit 处理分子结构与特征表示

在前三节里，我们处理的都是现成的数字和表格。但化学与材料研究真正面对的对象是**分子**——由原子和化学键连成的三维图结构。机器学习算法只认数字，不认识烧瓶里的粉末，也不直接认识化学结构式。因此，把 AI 用于分子与材料科学的第一步，永远是回答这个问题：**怎样把一个分子变成计算机能读懂、能计算的数据？**

这一节我们使用开源化学信息学标准工具包 **[RDKit](https://www.rdkit.org/)**，从最基础的分子字符串表示出发，亲手完成分子的解析、绘制、结构拆解、特征提取与子结构检索。

### 本节内容
1. SMILES 表示法、分子绘制与规范化（Canonical SMILES）
2. 拆解分子图：遍历原子、化学键、共轭体系与环
3. 把分子变成数字（一）：物理化学描述符（Descriptors）
4. 把分子变成数字（二）：Morgan 分子指纹、Tanimoto 相似度与 Murcko 骨架
5. 综合实战：用 RDKit + Pandas 清洗与检索 AqSolDB 数据库

In [ ]:
# colab-setup (本地运行时自动跳过)
import os, sys

if "google.colab" in sys.modules:
    _repo_dir = "/content/ai-for-materials"
    if not os.path.exists(_repo_dir):
        os.system("git clone --depth 1 https://github.com/Zeshengliu213/ai-for-materials.git " + _repo_dir)
    os.chdir(os.path.join(_repo_dir, "01 - Basics"))
    get_ipython().run_line_magic("pip", "install -q rdkit")


## 0. 环境检查与导入

本地环境如果尚未安装 RDKit，可以在终端运行 `pip install rdkit`；在 Google Colab 上运行下面这格代码会自动完成安装：

In [ ]:
try:
    import rdkit
except ImportError:
    %pip install -q rdkit
    import rdkit

from rdkit import Chem, DataStructs
from rdkit.Chem import Draw, Descriptors, rdFingerprintGenerator
from rdkit.Chem.rdMolDescriptors import CalcMolFormula
from rdkit.Chem.Scaffolds import MurckoScaffold

print("RDKit 版本:", rdkit.__version__)

---
## 1. SMILES、分子绘制与规范化

### 1.1 SMILES：把二维结构式压成一行字符

**SMILES**（Simplified Molecular Input Line Entry System）是目前化学数据库中最通用的线性分子表示法。只要记住下面几条简单规则，你就能直接读懂大多数常见有机分子的 SMILES：

| 语法规则 | 含义 | 示例 |
|---|---|---|
| **大写元素符号** | 普通（脂肪族）原子，相连的氢原子自动补齐 | `CCO`（乙醇）、`CCCCCC`（正己烷） |
| **小写元素符号** | **芳香体系**中的原子 | `c1ccccc1`（苯）、`c1ccsc1`（噻吩） |
| **`=` 与 `#`** | 双键、三键（单键和芳香键通常省略不写） | `C=C`（乙烯）、`C#N`（氰基） |
| **配对数字 `1...1`** | 成环标记：标有相同数字的两个原子首尾相连成环 | `C1CCCCC1`（环己烷） |
| **圆括号 `( )`** | 支链结构 | `CC(=O)O`（乙酸，羰基氧在支链上） |
| **方括号 `[ ]`** | 带电荷、指定同位素或显式带氢的特殊原子 | `[nH]`（吡咯/咔唑氮）、`[Si]`、`[Br-]` |

### 1.2 从 SMILES 到 RDKit 分子对象

在 Python 里，`"c1ccc2cc3cc4cc5ccccc5cc4cc3cc2c1"` 只是一个普通的文本字符串。调用 `Chem.MolFromSmiles(...)`，RDKit 就会按照化学价键规则把它解析成一个真正的**分子图对象（`Mol`）**。在 Jupyter Notebook 里，如果代码格最后一行是一个 `Mol` 对象，会自动渲染出它的化学结构式：

In [ ]:
pentacene = Chem.MolFromSmiles("c1ccc2cc3cc4cc5ccccc5cc4cc3cc2c1")
pentacene

如果传进去的 SMILES 违背了基本化学规则（比如写成了五碳化合价，或者把 5 个芳香碳强行围成一个五元环 `"c1cccc1"`，没法画出交替单双键的凯库勒式），RDKit 会怎样处理？

In [ ]:
bad_mol = Chem.MolFromSmiles("c1cccc1")    # 5 个芳香碳无法满足休克尔规则/凯库勒化
print("返回值:", bad_mol)

注意：遇到非法 SMILES 时，`Chem.MolFromSmiles` **不会中断程序抛异常**，而是打印一行红字警告并返回空值 **`None`**。因此，后面在批量处理几万个分子的数据集时，**第一步永远是检查并剔除返回值为 `None` 的脏数据**。

下面我们把有机电子学里最常打交道的 16 个代表性分子（涵盖噻吩类导电高分子单体、并苯类、BTBT 类高迁移率小分子、苝酰亚胺受体以及咔唑/三苯胺空穴传输材料）放进一个字典里，用 `Draw.MolsToGridImage` 一次性画成一张分子图谱：

In [ ]:
osc_smiles = {
    "thiophene": "c1ccsc1",
    "3-hexylthiophene": "CCCCCCc1ccsc1",
    "2,2'-bithiophene": "c1csc(c1)-c1cccs1",
    "EDOT": "C1COc2cscc2O1",
    "naphthalene": "c1ccc2ccccc2c1",
    "anthracene": "c1ccc2cc3ccccc3cc2c1",
    "tetracene": "c1ccc2cc3cc4ccccc4cc3cc2c1",
    "pentacene": "c1ccc2cc3cc4cc5ccccc5cc4cc3cc2c1",
    "TIPS-pentacene": "CC(C)[Si](C#Cc1c2cc3ccccc3cc2c(C#C[Si](C(C)C)(C(C)C)C(C)C)c2cc3ccccc3cc12)(C(C)C)C(C)C",
    "rubrene": "c1ccc(cc1)-c1c2ccccc2c(-c2ccccc2)c2c(-c3ccccc3)c3ccccc3c(-c3ccccc3)c12",
    "BTBT": "c1ccc2c(c1)sc1c2sc2ccccc21",
    "C8-BTBT": "CCCCCCCCc1ccc2c(c1)sc1c3ccc(CCCCCCCC)cc3sc21",
    "perylene": "c1cc2cccc3c4cccc5cccc(c(c1)c23)c54",
    "PTCDA": "O=C1OC(=O)c2ccc3c4ccc5C(=O)OC(=O)c6ccc(c7ccc1c2c37)c4c56",
    "carbazole": "c1ccc2c(c1)[nH]c1ccccc12",
    "triphenylamine": "c1ccc(cc1)N(c1ccccc1)c1ccccc1",
}

osc_mols = {name: Chem.MolFromSmiles(s) for name, s in osc_smiles.items()}

Draw.MolsToGridImage(
    list(osc_mols.values()),
    legends=list(osc_mols.keys()),
    molsPerRow=4,
    subImgSize=(250, 200),
)

### 🏋️ 练习 01d-1

从 `osc_mols` 中挑出并苯系列的 4 个分子（`"naphthalene"`, `"anthracene"`, `"tetracene"`, `"pentacene"`），用 `Draw.MolsToGridImage` 画成一行 4 列的图。要求每个分子下方的图例（`legends`）同时显示**分子名**和**分子式**（调用 `CalcMolFormula(mol)` 可以返回分子式字符串，中间用换行符 `\n` 连接，如 `"pentacene\nC22H14"`）。

In [ ]:
# 在这里写你的代码


### 1.3 规范化 SMILES（Canonical SMILES）：解决“同物异名”

细心的同学在看 SMILES 规则时可能会问：同一个噻吩环，从碳原子开始转圈写是 `"c1ccsc1"`，从硫原子开始写是 `"s1cccc1"`，写成单双键交替的凯库勒式则是 `"C1=CSC=C1"`。如果把这三个字符串直接按文本去比较，Python 会认为它们完全不同！

为了解决这个问题，我们可以先把任意写法的 SMILES 用 `Chem.MolFromSmiles` 读成分子图，再调用 **`Chem.MolToSmiles(mol)`** 重新导出。RDKit 内部有一套严格的原子编号排序算法，保证**只要是同一个拓扑结构的分子，导出的规范 SMILES 永远一模一样**：

In [ ]:
for s in ["c1ccsc1", "s1cccc1", "C1=CSC=C1"]:
    mol = Chem.MolFromSmiles(s)
    canon_s = Chem.MolToSmiles(mol)
    print(f"原始写法: {s:12s}  -->  规范 SMILES: {canon_s}")

> **为什么规范化 SMILES 在机器学习里至关重要？**
> 当我们从不同文献或数据库汇总实验数据时，同一个分子往往被写成了不同的 SMILES。如果不先做规范化就去重，同一个分子就会在数据集里出现两次——一旦其中一条分进了训练集、另一条分进了测试集，模型就等于“提前偷看了考题”（称为**数据泄漏 Data Leakage**），导致算出来的测试精度虚高（第 2 章会专门剖析这个问题）。

### 🏋️ 练习 01d-2

1. 写一个函数 `canonicalize(smiles)`：输入一个 SMILES 字符串，返回它的规范 SMILES；如果输入的 SMILES 无效（解析结果为 `None`），则返回 `None`。
2. 对下面 `collected` 列表里的 8 个 SMILES 调用你的函数，看看这 8 条记录里**实际上包含几个不同的分子**？（提示：Python 的集合 `set(列表)` 会自动去除重复元素。）

In [ ]:
collected = [
    "c1ccsc1", "C1=CSC=C1", "s1cccc1",
    "c1ccc2ccccc2c1", "C1=CC=C2C=CC=CC2=C1",
    "CCCCCCc1ccsc1", "c1cc(CCCCCC)cs1",
    "c1csc(c1)-c1cccs1",
]

# 在这里写你的代码


---
## 2. 拆解分子图：原子、化学键、共轭与环系

在 RDKit 内部，一个 `Mol` 对象其实是一张由**节点（原子 Atom）**和**边（化学键 Bond）**组成的数学图（这也是第 10 章图神经网络 GNN 的直接输入）。我们可以用 `mol.GetAtoms()` 和 `mol.GetBonds()` 遍历分子里的每一个原子和每一根键。

我们拿 **3-己基噻吩（`3-hexylthiophene`）** 来做个切片检查：

In [ ]:
m = osc_mols["3-hexylthiophene"]
print("重原子（非氢原子）总数:", m.GetNumAtoms())

for atom in m.GetAtoms():
    idx = atom.GetIdx()
    sym = atom.GetSymbol()
    arom = "芳香" if atom.GetIsAromatic() else "脂肪"
    deg = atom.GetDegree()          # 与几个重原子相连
    n_h = atom.GetTotalNumHs()      # 连着几个氢原子
    print(f"原子 {idx:2d}: {sym:2s} ({arom}) | 连接数={deg}, 氢原子数={n_h}")

再来看看它内部的每一根化学键。对于有机半导体而言，我们最关心的是哪些键处于 **$\pi$ 共轭体系（`GetIsConjugated()`）** 中：

In [ ]:
for bond in m.GetBonds():
    a1 = bond.GetBeginAtom().GetSymbol()
    a2 = bond.GetEndAtom().GetSymbol()
    btype = str(bond.GetBondType())
    conj = "[共轭]" if bond.GetIsConjugated() else ""
    print(f"键 {bond.GetIdx():2d}: {a1}-{a2} ({btype:8s}) {conj}")

上面的输出把“3-己基噻吩”的结构分工展现得一清二楚：前 5 根 C-C 单键属于正己基侧链，完全不共轭（绝缘、柔性，专门用来改善有机溶剂中的溶解度）；后 5 根键组成噻吩环，全部处于共轭体系中（提供载流子传输通道）。

我们还可以用 `mol.GetRingInfo()` 查询分子的环数与每个环的大小。另外要注意，RDKit 默认采用“隐式氢”（只把重原子作为图的节点）；如果需要计算包含氢原子在内的三维构象或量子化学输入，可以用 `Chem.AddHs(mol)` 把氢原子显式挂载上去：

In [ ]:
for name in ["pentacene", "rubrene", "BTBT"]:
    mol = osc_mols[name]
    ri = mol.GetRingInfo()
    ring_sizes = [len(r) for r in ri.AtomRings()]
    n_conj = sum(b.GetIsConjugated() for b in mol.GetBonds())
    print(f"{name:10s} | 总环数={ri.NumRings()}  各环大小={ring_sizes}  共轭键占比={n_conj}/{mol.GetNumBonds()}")

pentacene_with_h = Chem.AddHs(osc_mols["pentacene"])
print("\n并五苯默认重原子数:", osc_mols["pentacene"].GetNumAtoms(),
      "--> 显式加氢后总原子数:", pentacene_with_h.GetNumAtoms())

---
## 3. 把分子变成数字（一）：物理化学描述符

看懂了分子图之后，我们要解决核心问题：怎么把分子交给机器学习模型？

最直观的第一种办法是计算**分子描述符（Molecular Descriptors）**——也就是用确定性的经验公式或图算法，从分子结构直接算出一系列宏观或拓扑数值（如分子量 `MolWt`、脂水分配系数 `MolLogP`、极性表面积 `TPSA`、芳香环数 `NumAromaticRings`、可旋转键数 `NumRotatableBonds` 等）。

`01b` 和 `01c` 里 `solubility.csv` 的那 17 列特征，全都是用 `rdkit.Chem.Descriptors` 模块算出来的。下面我们亲手给刚才那 16 个有机半导体分子算一张描述符表：

In [ ]:
import pandas as pd

rows = []
for name, mol in osc_mols.items():
    rows.append({
        "name": name,
        "formula": CalcMolFormula(mol),
        "MolWt": Descriptors.MolWt(mol),
        "MolLogP": Descriptors.MolLogP(mol),
        "TPSA": Descriptors.TPSA(mol),
        "NumAromaticRings": Descriptors.NumAromaticRings(mol),
        "NumRotatableBonds": Descriptors.NumRotatableBonds(mol),
    })

osc_df = pd.DataFrame(rows)
osc_df.round(2)

对比一下表里的两组分子：
- **`pentacene` vs `TIPS-pentacene`**：芳香环数同样是 5，极性表面积 `TPSA` 都是 0，但接上两个三异丙基硅乙炔基（TIPS）后，可旋转键数 `NumRotatableBonds` 从 0 变成了 8，`MolLogP` 从 6.03 猛增到 11.61；
- **`BTBT` vs `C8-BTBT`**：接上两条辛基链后，可旋转键数从 0 变成 14，赋予了分子充分的构象柔性与亲油性，从而能够配成墨水进行大面积涂布。

除了上面挑出来的这 5 个，RDKit 的 `Descriptors.descList` 里一共内置了 **200 多个**不同的描述符，一行循环就能全部算出来（第 5 章做特征工程时我们会用到）：

In [ ]:
print(f"RDKit 共内置了 {len(Descriptors.descList)} 个分子描述符，前 10 个的名称：")
print([name for name, func in Descriptors.descList[:10]])

---
## 4. 把分子变成数字（二）：分子指纹、相似度与骨架

### 4.1 Morgan 分子指纹（ECFP）

描述符的优点是每个数字都有明确的物理化学含义，缺点是它只记录了整体的统计量，丢掉了具体的局部官能团连接细节。

**分子指纹（Molecular Fingerprint）** 换了一种编码思路：不预设哪些宏观性质重要，而是站在分子的每一个原子上，向外看半径 $r=1, 2, \dots$ 个化学键以内的**局部子结构环境**，然后通过哈希函数把每一种局部环境映射到一根长度为 2048 的二进制向量（0/1 数组）的某一位上。只要分子里出现过某个局部片段，对应的那一位就标为 `1`，否则为 `0`。

最著名的这类环形指纹叫做 **Morgan 指纹**（在药物与材料化学文献中也常称为 **ECFP**，当半径 `radius=2` 时对应 **ECFP4**）：

In [ ]:
fpgen = rdFingerprintGenerator.GetMorganGenerator(radius=2, fpSize=2048)

fp_pentacene = fpgen.GetFingerprint(osc_mols["pentacene"])
fp_tetracene = fpgen.GetFingerprint(osc_mols["tetracene"])

print("指纹向量总长度:", fp_pentacene.GetNumBits())
print("并五苯中被点亮为 1 的位数:", fp_pentacene.GetNumOnBits())
print("前 25 位的值:", list(fp_pentacene)[:25])

### 4.2 Tanimoto 相似度与二值指纹的盲区

有了两个分子的 0/1 指纹向量 $A$ 和 $B$，怎样量化这两个分子在结构上有多像？化学信息学里最通用的指标是 **Tanimoto 相似度（谷本系数，也即集合的 Jaccard 相似度）**：

$$T(A, B) = \frac{|A \cap B|}{|A \cup B|}$$

也就是“**两个分子都为 1 的位数**”除以“**至少有一个分子为 1 的位数**”。取值在 $0$（完全没有共同片段）到 $1$（指纹完全相同）之间。

我们来算一下并五苯（`pentacene`）和噻吩（`thiophene`）、以及并五苯和并四苯（`tetracene`）的 Tanimoto 相似度：

In [ ]:
fp_thiophene = fpgen.GetFingerprint(osc_mols["thiophene"])

print("pentacene vs thiophene :", round(DataStructs.TanimotoSimilarity(fp_pentacene, fp_thiophene), 3))
print("pentacene vs tetracene :", round(DataStructs.TanimotoSimilarity(fp_pentacene, fp_tetracene), 3))

看到第二行的结果了吗？**并五苯和并四苯的 Tanimoto 相似度竟然等于 `1.0`！**

它们明明一个是五环、一个是四环，能隙差了将近 $0.5\text{ eV}$，溶解度差了好几十倍，为什么在二值 Morgan 指纹（`radius=2`）眼里它们是“一模一样”的？

原因在于：半径为 2 的局部环境只看得到每个原子周围两步以内的邻居。无论并四苯还是并五苯，它们包含的局部环境种类是完全相同的（无非是“端环外侧的 CH”、“中间稠环的 CH”、“稠合桥头的季碳 C”这几种），**唯一的区别是中间稠环的片段在并五苯里多重复了一次**！而普通的 0/1 指纹只记录“有没有出现过这种片段”，不记录“出现了几次”。

> **贯穿整门课的核心启示**：
> **你选用的分子表示方法，决定了机器学习模型能看到什么、不能看到什么。**
> 如果只把 0/1 Morgan 指纹喂给模型去预测齐聚物或并苯的能隙，任何算法（哪怕是最先进的深度网络）都无法区分并四苯和并五苯！
>
> 要解决这个盲区，可以改用记录片段出现次数的**计数指纹（`GetCountFingerprint`）**，或者把反映全局尺寸的物理化学描述符（如 `MolWt`、`NumAromaticRings`）与指纹拼接在一起：

In [ ]:
cfp_pentacene = fpgen.GetCountFingerprint(osc_mols["pentacene"])
cfp_tetracene = fpgen.GetCountFingerprint(osc_mols["tetracene"])

print("改用计数指纹后 pentacene vs tetracene 相似度:",
      round(DataStructs.TanimotoSimilarity(cfp_pentacene, cfp_tetracene), 3))

我们用 `DataStructs.BulkTanimotoSimilarity` 一次性算出那 16 个有机电子分子两两之间的相似度矩阵，并用 `seaborn.heatmap` 画出来看看：

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

names = list(osc_mols.keys())
fps = [fpgen.GetFingerprint(osc_mols[n]) for n in names]
sim_df = pd.DataFrame(
    [DataStructs.BulkTanimotoSimilarity(fp, fps) for fp in fps],
    index=names,
    columns=names,
)

plt.figure(figsize=(8.5, 7))
sns.heatmap(sim_df, cmap="viridis", vmin=0, vmax=1, square=True)
plt.title("Pairwise Tanimoto similarity (Morgan, r=2)")
plt.show()

<!-- ai4m-style -->
> **📘 课程统一绘图风格**：下面这格加载课程自带的 `ai4m.mplstyle`（配色与课程网站同源，无上右边框，仅横向浅网格，保存默认 300 dpi）。`C` 是课程配色，手动指定颜色时写 `color=C.crimson`。


In [ ]:
# ai4m-style：课程统一绘图风格（本地与 Colab 通用）
import pathlib, sys

sys.path.insert(0, str(pathlib.Path.cwd().parent))   # course/ 根目录
import ai4m_style as ai4m

C = ai4m.use()        # 应用 ai4m.mplstyle，返回课程配色


### 🏋️ 练习 01d-3

1. 从上面的 `sim_df` 表格中，找出除了 `BTBT` 自身以外，与 **`BTBT` 相似度最高的前 5 个分子**及其相似度数值（提示：`sim_df["BTBT"].drop("BTBT").sort_values(ascending=False).head(5)`）。
2. 思考一下：`C8-BTBT` 和 `BTBT` 的共轭核心骨架一模一样，为什么它们俩的 Tanimoto 相似度却只有 0.30 左右？

In [ ]:
# 在这里写你的代码


### 4.3 提取分子核心：Bemis–Murcko 骨架

在刚才的练习 01d-3 里你已经发现：当我们在共轭核上接上长长的烷基链或硅烷基链（比如 `C8-BTBT` 或 `TIPS-pentacene`）时，侧链会产生大量脂肪族局部片段，把全分子的 Tanimoto 相似度大幅拉低。

如果我们想让计算机忽略外围的可溶性侧链，直接回答“**这个分子的核心骨架是什么**”，可以使用 RDKit 的 **`MurckoScaffold.GetScaffoldForMol(mol)`**。它会自动剪掉所有悬空的末端侧链，只保留环系以及连接环与环之间的主干通道——这恰好对应了有机光电材料中“**共轭核 + 侧链**”的分子设计逻辑：

In [ ]:
sample_names = ["3-hexylthiophene", "TIPS-pentacene", "C8-BTBT", "rubrene", "triphenylamine"]
grid_mols, grid_legends = [], []

for name in sample_names:
    mol = osc_mols[name]
    core = MurckoScaffold.GetScaffoldForMol(mol)
    grid_mols.extend([mol, core])
    grid_legends.extend([name, "Scaffold: " + Chem.MolToSmiles(core)])

Draw.MolsToGridImage(grid_mols, legends=grid_legends, molsPerRow=2, subImgSize=(320, 200))

你看，`3-hexylthiophene` 剃掉己基后还原成了噻吩，`TIPS-pentacene` 剃掉硅烷基后还原成了并五苯，`C8-BTBT` 则完美还原成了 `BTBT`。

> **伏笔（骨架划分 Scaffold Split）**：在第 2 章评估机器学习模型时，如果随机划分训练集和测试集，`BTBT` 和 `C8-BTBT` 很容易分到两边，让模型靠死记硬背骨架拿高分。更严格的考法是按 **Murcko 骨架** 分组（把相同骨架的衍生物捆绑在同一侧），检验模型能不能真正推广到从未见过的新骨架上。

---
## 5. 综合实战：用 RDKit + Pandas 清洗与检索 AqSolDB

### 5.1 批量解析分子与脏数据排查

现在我们把 `01b` 学的 Pandas 和刚刚学的 RDKit 结合起来，对 `solubility.csv` 里的全部 9482 个化合物做一次真实的化学信息学清洗。

In [ ]:
import os

DATA_FILE = "data/solubility.csv"
if not os.path.exists(DATA_FILE):
    DATA_FILE = "solubility.csv"
if not os.path.exists(DATA_FILE):
    try:
        from google.colab import files
        print("请上传 solubility.csv")
        files.upload()
    except ImportError:
        raise FileNotFoundError("找不到 solubility.csv，请确保它在 data/ 目录下")

df = pd.read_csv(DATA_FILE)
print(f"读取成功！原始记录共 {len(df)} 条。")

我们用 `df["SMILES"].apply(Chem.MolFromSmiles)` 把每一行的 SMILES 字符串转换成 RDKit 分子对象，存进新的一列 `df["mol"]`，并立刻检查有没有解析失败变成 `None`（在 Pandas 里表现为缺失值 `NaN`）的记录：

In [ ]:
from rdkit import RDLogger
RDLogger.DisableLog("rdApp.warning")   # 关闭无关紧要的警告刷屏，只保留错误提示

df["mol"] = df["SMILES"].apply(Chem.MolFromSmiles)
print("解析失败 (返回 None) 的条目数:", df["mol"].isna().sum())

# 看看是哪两个分子解析失败了：
df[df["mol"].isna()][["Name", "SMILES"]]

果然抓出了两条无法解析的“脏数据”！仔细看它们的 SMILES，里面出现了 `[NH++]`、`[CH-]` 等不合理的形式电荷写法，导致 RDKit 无法确定芳环的电子数。在公开数据集里遇到这种格式瑕疵非常普遍。我们用 `.dropna(subset=["mol"])` 把这两行剔除，防止后面调用分子方法时报错：

In [ ]:
df = df.dropna(subset=["mol"]).reset_index(drop=True)
print(f"剔除无效结构后，剩余 {len(df)} 个有效分子。")

有了 `df["mol"]` 这一列，我们就可以随时用 `.apply(...)` 算出任何想要的分子特征。比如我们自己用 `Descriptors.MolWt` 算一遍分子量，再调用分子对象的 `.GetNumAtoms()` 方法算一下重原子数，跟表里原有的列核对一下：

In [ ]:
df["MolWt_rdkit"] = df["mol"].apply(Descriptors.MolWt)
df["n_heavy"] = df["mol"].apply(lambda m: m.GetNumAtoms())

df[["Name", "MolWt", "MolWt_rdkit", "HeavyAtomCount", "n_heavy"]].head()

> **顺便认识一下 `lambda` 匿名函数**：
> 上面第二行里的 `lambda m: m.GetNumAtoms()` 是 Python 里写“一次性小函数”的简写形式，意思完全等价于：
> ```python
> def get_n_atoms(m):
>     return m.GetNumAtoms()
> ```
> 在配合 Pandas 的 `.apply()` 做简单转换时，用 `lambda 输入参数: 返回表达式` 可以省去专门 `def` 起名字的麻烦。

### 🏋️ 练习 01d-4

1. 验证我们刚算出来的 `MolWt_rdkit` 与原表中的 `MolWt` 是否一致：计算两列之差的绝对值的最大值 `(df["MolWt"] - df["MolWt_rdkit"]).abs().max()`。
2. 回头看 `solubility.csv` 的第一行，它的 SMILES 是 `"[Br-].CCCCCCCCCCCCCCCCCC[N+](C)(C)C"`（十六烷基三甲基溴化铵）。在 SMILES 语法里，圆点 **`.`** 代表**互不以共价键相连的多个独立组分**（比如有机盐的阴阳离子、结晶水合物或混合物）。请统计一下，在这 9480 个条目里，SMILES 中包含 `.` 的条目一共有多少个？如果我们要训练一个预测“中性有机分子水溶解度”的模型，这批含 `.` 的数据会带来什么影响？

In [ ]:
# 在这里写你的代码


### 5.2 子结构检索（Substructure Search）

在 `01b` 里，我们曾试着用名称字符串 `df["Name"].str.contains("thiophene")` 去找含噻吩的分子。但很多化合物用的是俗名或商品名，名字里根本没有 `thiophene`；就算去搜 SMILES 字符串，同一个噻吩环在复杂稠环里的写法也千变万化。

真正严谨的做法是**基于分子图的子结构匹配**：用 `Chem.MolFromSmarts("c1ccsc1")` 定义一个结构片段查询模式（SMARTS 是 SMILES 的查询扩展语法），然后用 `mol.HasSubstructMatch(模式)` 让 RDKit 在分子图里做子图同构搜索。不管噻吩环藏在哪个角落、怎么编号，都能一网打尽：

In [ ]:
thiophene_pattern = Chem.MolFromSmarts("c1ccsc1")

df["has_thiophene"] = df["mol"].apply(lambda m: m.HasSubstructMatch(thiophene_pattern))
hits = df[df["has_thiophene"]]
print(f"全库共检索到 {len(hits)} 个含有噻吩环的化合物。前 12 个结构如下：")

Draw.MolsToGridImage(
    hits["mol"].head(12).tolist(),
    legends=hits["Name"].head(12).str[:24].tolist(),
    molsPerRow=4,
    subImgSize=(250, 190),
)

如果不仅想知道“含不含”，还想知道匹配到的子结构具体落在哪几个原子上，可以调用 `mol.GetSubstructMatches(模式)`，并在画图时把这些原子高亮出来：

In [ ]:
bithiophene = osc_mols["2,2'-bithiophene"]
matches = bithiophene.GetSubstructMatches(thiophene_pattern)
print("匹配到的噻吩环原子序号:", matches)

# 把匹配到的所有原子序号展平放进一个列表，传给 highlightAtoms
highlight_ids = [idx for match in matches for idx in match]
Draw.MolToImage(bithiophene, highlightAtoms=highlight_ids, size=(300, 200))

### 🏋️ 练习 01d-5（本章主线验证）

现在我们用子结构搜索，为第一部分“大 $\pi$ 共轭核难溶”的主线画上句号：
1. 用 SMARTS 字符串 `"c1ccc2cc3ccccc3cc2c1"` 构建一个**蒽核**（三个线性稠合苯环）的查询模式，找出 `df` 中所有含有蒽核子结构的化合物，看看一共有多少个？
2. 用 `df.groupby(...)["Solubility"].mean()` 对比“含蒽核”和“不含蒽核”两组化合物的平均 $\log S$。含蒽核的分子平均溶解度比普通分子低了多少个数量级？
3. （选做）用 `sns.histplot(..., stat="density", common_norm=False, element="step")` 画出这两组分子的 $\log S$ 概率密度分布对比图。

In [ ]:
# 在这里写你的代码


---
## 第一部分总结

回顾 `01a` 到 `01d` 这四个 Notebook，你已经走完了材料与化学数据科学的完整基本功闭环：
1. **Python 语法与数组运算（`01a`）**：变量、容器、条件循环、函数封装与 NumPy 向量化计算；
2. **表格数据清洗与聚合（`01b`）**：用 Pandas 读入近万条真实溶解度数据，完成筛选、排序、分组统计与多表关联合并；
3. **统计可视化与探索性分析（`01c`）**：用 Matplotlib 与 Seaborn 绘制分布、散点、箱线图与相关性热图，识别出经验外推风险与特征共线性；
4. **化学信息学与分子特征工程（`01d`）**：用 RDKit 解析并规范化 SMILES，拆解共轭键与环系，提取 Murcko 骨架，计算物理化学描述符与 Morgan 指纹，并通过子结构检索证实了稠环共轭核对溶解度的显著压制。

**下一章预告（`02 - Supervised Learning`）**：
既然我们已经能把每一个分子变成一行描述符或一串 2048 维的指纹向量，接下来就正式进入**机器学习建模**——我们将用线性回归、正则化模型、随机森林与梯度提升树来预测分子的溶解度，并深入探讨交叉验证、骨架划分（Scaffold Split）与模型适用域（Applicability Domain）。